In [ ]:
import polars as pl
from pathlib import Path


def contrast_direction(lo: float | None, hi: float | None) -> str | None:
    if lo is None or hi is None:
        return None
    if lo > 1.0:
        return "divergent longer"
    if hi < 1.0:
        return "convergent longer"
    return "n.s."

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "protists_mit",
    "green_algae_mit",
    "plants_plt",
    "protists_plt",
    "green_algae_plt",
]

PRIMARY_MODEL = "both (adjusted mean; PRIMARY)"

In [ ]:
rows_s4 = []

for group in groups:
    gdir = BASE / group
    contrast = (
        pl.read_csv(
            gdir / "brms_type_length" / "brms_contrast_row.tsv",
            separator="\t",
            separator="\t",
        )
        .filter(pl.col("model") == PRIMARY_MODEL)
        .row(0, named=True)
    )

    rows_s4.append(
        {
            "group": group,
            "IGRs": contrast["N_regions"],
            "ratio_div_over_conv": contrast["ratio_div_over_conv"],
            "ratio_lo": contrast["ratio_lo"],
            "ratio_hi": contrast["ratio_hi"],
            "P_div_gt_conv": contrast["P_div_gt_conv"],
            "contrast_direction": contrast_direction(
                contrast["ratio_lo"],
                contrast["ratio_hi"],
            ),
        }
    )

s4 = pl.DataFrame(rows_s4)

In [ ]:
s4 = s4.with_columns(
    [
        pl.col("IGRs").cast(pl.Int64),
        pl.col("ratio_div_over_conv").cast(pl.Float64).round(3),
        pl.col("ratio_lo").cast(pl.Float64).round(3),
        pl.col("ratio_hi").cast(pl.Float64).round(3),
        pl.col("P_div_gt_conv").cast(pl.Float64).round(4),
    ]
)

s4_formatted = pl.DataFrame(rows_s4).with_columns(
    [
        pl.format(
            "{} ({}, {})",
            pl.col("ratio_div_over_conv").round(3),
            pl.col("ratio_lo").round(3),
            pl.col("ratio_hi").round(3),
        ).alias("Divergent/convergent length ratio (95% CrI)"),
        pl.when(pl.col("P_div_gt_conv").is_null())
        .then(None)
        .when(pl.col("P_div_gt_conv") >= 0.99995)
        .then(pl.lit(">0.9999"))
        .when(pl.col("P_div_gt_conv") <= 0.00005)
        .then(pl.lit("<0.0001"))
        .otherwise(pl.col("P_div_gt_conv").round(4).cast(pl.String))
        .alias("P(divergent > convergent)"),
    ]
)
s4_formatted.write_csv(BASE / "code" / "supplementary_table4.tsv", separator="\t")